In [ ]:
import pandas as pd
import numpy as np
import re
# from spellchecker import SpellChecker
RANDOM_SEED = 577
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from collections import defaultdict
import gensim.downloader as api
from torch.nn.utils.rnn import pad_sequence
import openai
import tqdm
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def preprocess(text):
    # Replace repeated punctuation signs with labels and add spaces
    text = re.sub(r'(\.{2,})', r' multistop ', text)
    text = re.sub(r'(\!{2,})', r' multiexclamation ', text)
    text = re.sub(r'(\?{2,})', r' multiquestion ', text)
    # Add spaces before and after single punctuation signs
    text = re.sub(r'(\.|\!|\?|\,)', r' ', text)
    return text
def load_emoticons(emo_filename):
    # Load emoticons and their polarity from a file
    emoticon_dict = {}
    with open(emo_filename, 'r', encoding='latin-1') as file:
        for line in file:
            emoticon, polarity = line.strip().split('\t')
            emoticon_dict[emoticon] = polarity
    return emoticon_dict

# Load emoticons and their polarity from a file
emoticon_dict = load_emoticons('EmoticonLookupTable.txt')

def replace_emoticons(text, emoticon_dict=emoticon_dict):
    # Replace emoticons with their polarity and delete neutral ones
    for emoticon, polarity in emoticon_dict.items():
        pattern = re.compile(re.escape(emoticon), re.IGNORECASE)
        if polarity == '1':
            text = pattern.sub("positive", text)
        elif polarity == '-1':
            text = pattern.sub("negative", text)
        else:
            text = pattern.sub('', text)
            
    text = re.sub(r'[^a-zA-Z\s]', '', text)
            
    return text.split()

In [ ]:
c = pd.read_csv("tweets_preprocess_emoji.csv",header=None)
df = pd.read_csv("tweets_nopreprocess.csv")
df["text"] = df["text"].apply(lambda x: re.sub('@[^\s]+','',x))
text_pd = pd.DataFrame()
text_pd["text"] = df["text"][0:34825]
text_pd["text"] = text_pd["text"].apply(lambda x: re.sub(r'[^\w\s]','',x))
# text_pd["text"][3]

In [ ]:
text_pd["text"] = text_pd["text"].apply(lambda x: re.sub('[A-Z]', lambda y: y.group(0).lower(), x))
text_pd["text"] = text_pd["text"].apply(lambda x: re.sub(r'\d+', '', x))


text_pd["emoji"] = c[1]
emoji_map = {}
counter = 0
for i in text_pd["emoji"]:
  if i not in emoji_map:
    emoji_map[i] = counter
    counter+=1
text_pd["emoji_id"] = text_pd["emoji"].apply(lambda x: emoji_map[x])
text_pd["text"] = text_pd["text"].apply(lambda x: x.strip())
text_pd["text"] = text_pd["text"].apply(lambda x: re.sub(' +', ' ', x).strip())

emoji_map

In [ ]:
df = text_pd.groupby(by="emoji", as_index=False).count()
emoji_used = df.sort_values(by="text", ascending=False).reset_index(drop=True)["emoji"][:100].tolist()

text_pd = text_pd[text_pd["emoji"].isin(emoji_used)]
emoji_map = {}
counter = 0
for i in text_pd["emoji"]:
  if i not in emoji_map:
    emoji_map[i] = counter
    counter+=1
text_pd["emoji_id"] = text_pd["emoji"].apply(lambda x: emoji_map[x])

text_pd = text_pd[:].reset_index(drop=True)
print(text_pd)


In [ ]:
df

In [ ]:
def get_text(text):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": f"Text: {text}"}
            ],
        )
        answer = response['choices'][0]['message']['content'].strip()
        return answer
    except Exception as e:
        print(e)
        return ""

In [ ]:
text_pd[text_pd["emoji"]==emoji_used[0]] 
for i in tqdm.tqdm(emoji_used):
    value = df[df["emoji"] == i]["text"].values[0]
    if(value < 200):
        emoji = i
        temp_df = text_pd[text_pd["emoji"]==emoji]
        temp_df = temp_df.text.to_list()
        augmentated = []
        while(len(temp_df) < 200):
            print(len(temp_df))
            text = np.random.choice(temp_df,1)
            a = get_text(f"create 30 sentences with the same meaning or similar meaning: {text}")
            for k in a.split('\n'):
                if len(k.split('. ')) > 1:
                    to_append = k.split('. ')[1]
                    augmentated.append(to_append)
                    temp_df.append(to_append)
            # augmentated.extend([k.split('. ')[1] for k in a.split('\n')])
            # temp_df.extend([k.split('. ')[1] for k in a.split('\n')])
            
        for j in augmentated:
            df2 = {'text': j, 'emoji': emoji,'emoji_id':emoji_map[emoji]}
            text_pd = text_pd.append(df2, ignore_index = True)

In [ ]:
augmentated

In [ ]:
text_pd.to_csv('tweets11111.csv')